# Multi-Node ML Job — Minimal Working Example

This notebook demonstrates a multi-node distributed XGBoost training job using Snowflake ML Jobs.

**Requirements:**
- A compute pool with `MAX_NODES >= target_instances`
- A stage for ML Job artifacts
- The `snowflake-ml-python` package (pre-installed on Container Runtime)

**Key points:**
- ML Jobs use `target_instances=N` for multi-node (NOT `scale_cluster()`)
- Inner functions passed to Tuner must capture variables explicitly
- The distributed APIs work identically on 1 or N nodes — just change `target_instances`

In [ ]:
# Step 1: Setup
from snowflake.snowpark.context import get_active_session
session = get_active_session()

DB = "RRD_ML_DEMO"
SCHEMA = "DISTRIBUTED_TRAINING"

session.sql(f"CREATE DATABASE IF NOT EXISTS {DB}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}.{SCHEMA}").collect()
session.sql(f"CREATE STAGE IF NOT EXISTS {DB}.{SCHEMA}.ML_JOB_STAGE").collect()

print(f"Setup complete: {DB}.{SCHEMA}")

In [ ]:
# Step 2: Create a small test dataset
session.sql(f"""
CREATE OR REPLACE TABLE {DB}.{SCHEMA}.MULTINODE_TEST_DATA AS
SELECT
    ROW_NUMBER() OVER (ORDER BY SEQ4()) AS ID,
    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(1)) AS FEAT_0,
    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(2)) AS FEAT_1,
    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(3)) AS FEAT_2,
    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(4)) AS FEAT_3,
    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(5)) AS FEAT_4,
    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(6)) AS FEAT_5,
    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(7)) AS FEAT_6,
    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(8)) AS FEAT_7,
    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(9)) AS FEAT_8,
    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(10)) AS FEAT_9,
    IFF(UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(99)) > 0.5, 1, 0) AS TARGET
FROM TABLE(GENERATOR(ROWCOUNT => 100000))
""").collect()

print("Created test table: 100K rows x 10 features")

In [ ]:
# Step 3: Define the multi-node ML Job
#
# KEY CONFIGURATION:
#   - target_instances=3  → provisions 3 nodes automatically
#   - No scale_cluster() needed inside ML Jobs
#   - Compute pool MAX_NODES must be >= 3

from snowflake.ml.jobs import remote

COMPUTE_POOL = "CREDIT_RISK_TRAINING_POOL"  # Change to your compute pool name

@remote(
    COMPUTE_POOL,
    stage_name=f"{DB}.{SCHEMA}.ML_JOB_STAGE",
    session=session,
    target_instances=3,  # <-- This enables multi-node
)
def multinode_xgb_job(table_fqn: str):
    """Minimal multi-node XGBoost training job."""
    from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
    from snowflake.ml.data.data_connector import DataConnector
    from snowflake.snowpark.context import get_active_session
    from sklearn.metrics import roc_auc_score
    import pandas as pd
    import time

    session = get_active_session()

    # Load data
    df = session.table(table_fqn)
    feature_cols = [c for c in df.columns if c.startswith("FEAT_")]
    label_col = "TARGET"
    train_df, test_df = df.random_split([0.8, 0.2], seed=42)

    # Configure distributed training
    # num_workers=-1 means auto-detect based on available nodes
    scaling_config = XGBScalingConfig(
        num_workers=-1,
        num_cpu_per_worker=-1,
        use_gpu=False,
    )

    estimator = XGBEstimator(
        n_estimators=100,
        params={
            "max_depth": 6,
            "learning_rate": 0.1,
            "tree_method": "hist",
            "objective": "binary:logistic",
            "eval_metric": "auc",
        },
        scaling_config=scaling_config,
    )

    # Train (distributed across all nodes automatically)
    train_connector = DataConnector.from_dataframe(train_df)
    start = time.time()
    model = estimator.fit(train_connector, input_cols=feature_cols, label_col=label_col)
    train_time = time.time() - start

    # Predict (single-node for small test set)
    import xgboost
    test_pd = test_df.to_pandas()
    dtest = xgboost.DMatrix(test_pd[feature_cols])
    y_prob = model.predict(dtest)
    auc = roc_auc_score(test_pd[label_col].astype(int), y_prob)

    return {
        "train_time_secs": round(train_time, 1),
        "auc": round(auc, 4),
        "num_features": len(feature_cols),
        "train_rows": int(train_df.count()),
    }

print(f"ML Job defined: multinode_xgb_job")
print(f"  Compute Pool: {COMPUTE_POOL}")
print(f"  Target Instances: 3")

In [ ]:
# Step 4: Set session context and submit the job
session.sql(f"USE DATABASE {DB}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

job = multinode_xgb_job(
    table_fqn=f"{DB}.{SCHEMA}.MULTINODE_TEST_DATA"
)

print(f"Job submitted: {job.id}")
print(f"Status: {job.status}")
print(f"")
print("Monitor:")
print(f"  job.status              → current status")
print(f"  job.wait()              → block until complete")
print(f"  job.result()            → return value")
print(f"  job.get_logs()          → head node logs")
print(f"  job.get_logs(instance_id=1) → worker logs")

In [ ]:
# Step 5: Wait for completion and get results
job.wait()
result = job.result()

print(f"Job completed! Status: {job.status}")
print(f"")
print(f"Results:")
print(f"  Training time: {result['train_time_secs']}s")
print(f"  AUC-ROC:       {result['auc']}")
print(f"  Features:      {result['num_features']}")
print(f"  Train rows:    {result['train_rows']:,}")
print(f"")
print("--- Logs (last 1000 chars) ---")
print(job.get_logs()[-1000:])

## Multi-Node HPO (with Tuner)

The key difference from single-node: variables used inside `train_func` must be **explicitly captured** before the function definition. This is because the Tuner serializes `train_func` and sends it to worker processes.

In [ ]:
from snowflake.ml.jobs import remote

DB = "RRD_ML_DEMO"
SCHEMA = "DISTRIBUTED_TRAINING"

@remote(
    compute_pool="CREDIT_RISK_INFERENCE_POOL",
    stage_name=f"{DB}.{SCHEMA}.ML_JOB_STAGE",
    session=session,
    target_instances=5,
)
def multinode_hpo_job(table_fqn: str, num_trials: int = 6):
    """Multi-node HPO with materialized data and distributed XGBEstimator."""
    from snowflake.ml.modeling.tune import Tuner, TunerConfig, get_tuner_context, uniform, choice, randint
    from snowflake.ml.modeling.tune.search import RandomSearch
    from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
    from snowflake.ml.data.data_connector import DataConnector
    from snowflake.snowpark.context import get_active_session
    from sklearn.metrics import roc_auc_score
    import pandas as pd
    import time

    print("Starting HPO job...")
    session = get_active_session()

    # Explicitly activate warehouse to avoid silent hang
    print("Activating warehouse...")
    session.sql("USE WAREHOUSE ML_DEMO_WH").collect()
    print("Warehouse active.")

    # Load and split data
    print(f"Loading data from {table_fqn}...")
    df = session.table(table_fqn)
    feature_cols = [c for c in df.columns if c.startswith("FEAT_")]
    label_col = "TARGET"
    print(f"  Features: {len(feature_cols)}, Label: {label_col}")

    train_df, test_df = df.random_split([0.8, 0.2], seed=42)

    # Materialize into temp tables BEFORE creating DataConnectors.
    # This ensures the Tuner doesn't need a live warehouse connection
    # during trial execution (which can cause silent hangs).
    print("Materializing train/test splits into temp tables...")
    train_df.write.mode("overwrite").save_as_table(
        f"{DB}.{SCHEMA}.HPO_TRAIN_TEMP", table_type="temporary"
    )
    test_df.write.mode("overwrite").save_as_table(
        f"{DB}.{SCHEMA}.HPO_TEST_TEMP", table_type="temporary"
    )
    train_count = session.table(f"{DB}.{SCHEMA}.HPO_TRAIN_TEMP").count()
    test_count = session.table(f"{DB}.{SCHEMA}.HPO_TEST_TEMP").count()
    print(f"  Train rows: {train_count:,}, Test rows: {test_count:,}")

    # Create DataConnectors from materialized tables
    dataset_map = {
        "train": DataConnector.from_dataframe(session.table(f"{DB}.{SCHEMA}.HPO_TRAIN_TEMP")),
        "test": DataConnector.from_dataframe(session.table(f"{DB}.{SCHEMA}.HPO_TEST_TEMP")),
    }

    search_space = {
        "n_estimators": choice([50, 100, 150]),
        "max_depth": randint(4, 10),
        "learning_rate": uniform(0.01, 0.2),
    }

    tuner_config = TunerConfig(
        metric="auc",
        mode="max",
        search_alg=RandomSearch(random_state=42),
        num_trials=num_trials,
        max_concurrent_trials=1,
    )

    # Capture variables for closure serialization
    _feature_cols = feature_cols
    _label_col = label_col

    def train_func():
        from snowflake.ml.modeling.tune import get_tuner_context
        from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
        from sklearn.metrics import roc_auc_score
        import pandas as pd

        tuner_context = get_tuner_context()
        config = tuner_context.get_hyper_params()
        dm = tuner_context.get_dataset_map()

        estimator = XGBEstimator(
            n_estimators=int(config["n_estimators"]),
            params={
                "max_depth": int(config["max_depth"]),
                "learning_rate": config["learning_rate"],
                "tree_method": "hist",
                "objective": "binary:logistic",
                "eval_metric": "auc",
            },
            scaling_config=XGBScalingConfig(
                num_workers=3,
                use_gpu=False,
            ),
        )

        booster = estimator.fit(dm["train"], input_cols=_feature_cols, label_col=_label_col)

        predictions = estimator.predict(dm["test"])
        pred_df = predictions if isinstance(predictions, pd.DataFrame) else predictions.to_pandas()
        pred_col = [c for c in pred_df.columns if "predict" in c.lower()][0]

        if _label_col in pred_df.columns:
            target_col = _label_col
        elif _label_col.upper() in pred_df.columns:
            target_col = _label_col.upper()
        else:
            target_col = _label_col

        auc = roc_auc_score(pred_df[target_col].astype(int), pred_df[pred_col])
        tuner_context.report(metrics={"auc": auc}, model=booster)

    print(f"Starting Tuner with {num_trials} trials...")
    start = time.time()
    tuner = Tuner(train_func, search_space, tuner_config)
    results = tuner.run(dataset_map=dataset_map)
    elapsed = time.time() - start

    best_auc = float(results.best_result["auc"].iloc[0])
    print(f"\nHPO Complete in {elapsed:.1f}s")
    print(f"Best AUC: {best_auc:.4f}")
    return {"best_auc": round(best_auc, 4), "num_trials": num_trials, "elapsed_secs": round(elapsed, 1)}


# Submit the job
job = multinode_hpo_job(f"{DB}.{SCHEMA}.MULTINODE_TEST_DATA", num_trials=6)
print(f"Job submitted: {job.id}")
print(f"Status: {job.status}")

In [ ]:
# Step 7: Submit HPO job
session.sql(f"USE DATABASE {DB}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

hpo_job = multinode_hpo_job(
    table_fqn=f"{DB}.{SCHEMA}.MULTINODE_TEST_DATA",
    num_trials=1,
)

print(f"HPO Job submitted: {hpo_job.id}")
print(f"Status: {hpo_job.status}")

In [ ]:
# Step 8: Get HPO results
hpo_job.wait()
hpo_result = hpo_job.result()

print(f"HPO Job completed! Status: {hpo_job.status}")
print(f"  Best AUC: {hpo_result['best_auc']}")
print(f"  Trials:   {hpo_result['num_trials']}")
print(f"")
print("--- Logs ---")
print(hpo_job.get_logs()[-1500:])

## Troubleshooting Multi-Node

| Symptom | Cause | Fix |
|---------|-------|-----|
| Job runs on 1 node only | `target_instances=1` or pool `MAX_NODES=1` | Set `target_instances=N` and pool `MAX_NODES >= N` |
| `NameError` in worker logs | Variables not captured in `train_func` closure | Assign to `_var = var` before `def train_func()` |
| Workers pending / never start | `num_cpu_per_worker` exceeds available CPUs | Use `-1` for auto-detect or check node CPU count |
| Training slow despite multi-node | Too many small workers (high communication overhead) | Use fewer workers with more CPUs: `num_workers=N, num_cpu_per_worker=-1` |
| `scale_cluster` error in ML Job | `scale_cluster()` is for notebooks only | Remove it — ML Jobs handle scaling via `target_instances` |
| Stale config after cancel | Old Ray processes hold resources | Restart kernel (notebook) or resubmit job (ML Job) |